In [1]:
import requests
from bs4 import BeautifulSoup
import sqlite3

url="https://internshala.com/internships/"
response=requests.get(url)
print(response.status_code)
if response.status_code != 200:
    print("Failed to Login Page Error" ,response.status_code)

200


In [2]:
soup = BeautifulSoup(response.text, "html.parser")
jobs = soup.find_all("div", class_="individual_internship")
print(f" In This website I found {len(jobs)} job listing")

 In This website I found 51 job listing


In [3]:
con=sqlite3.connect("internships.db")
cursor=con.cursor()

cursor.execute("""
    CREATE  TABLE IF NOT EXISTS internships(
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        title    TEXT,
        company  TEXT,
        location TEXT,
        stipend  TEXT,
        duration  TEXT,
        link     TEXT
        )
    """)

cursor.execute("DELETE FROM internships")
print("Table Created")


Table Created


In [4]:
def get_text(tag):
    if tag:
        return tag.get_text(separator=" ").strip()
    return "Not Satisfied"

def stipend_value(stipend_text):
    number = ""
    for ch in stipend_text:
        if ch.isdigit():
            number += ch
        elif number:   # ← pehla number milne ke baad ruk jao
            break
    return int(number) if number else 0

def pass_Filter(title,company,location,stipend,duration):
    if FILTER_LOCATION and FILTER_LOCATION.lower()  not in location.lower():
        return False
    if FILTER_STIPEND >0:
        stipend_value=stipend_S(stipend)
        if stipend_value < FILTER_STIPEND:
            return False

    if FILTER_DURATION and FILTER_DURATION.lower() not in duration.lower():
        return False
    return True



In [5]:
result = []
rejected = 0
saved = 0

FILTER_LOCATION = input("Enter location: ")
FILTER_STIPEND  = int(input("Enter min stipend: "))
FILTER_DURATION = input("Enter the duration: ")

import time

for page in range(1, 6):

    url = f"https://internshala.com/internships/page-{page}"

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    jobs = soup.find_all("div", class_="individual_internship")

    print(f"Page {page}: {len(jobs)} internships found")

    for job in jobs:

        title_tag = job.find("a", class_="job-title-href")

        # ✅ Bug 2 Fix — double prefix check
        if title_tag and title_tag.get("href"):
            href = title_tag["href"]
            link = href if href.startswith("http") else "https://internshala.com" + href
        else:
            link = ""

        company_tag = job.find(
            "p",
            class_="company-name"
        )

        location_tag = job.find(
            "div",
            class_="row-1-item locations"
        )

        stipend_tag = job.find(
            "span",
            class_="stipend"
        )

        duration_tag = job.find(
            "i",
            class_="ic-16-calendar"
        )

        title    = get_text(title_tag)
        company  = get_text(company_tag)
        location = get_text(location_tag)
        stipend  = get_text(stipend_tag)

        duration = (
            get_text(duration_tag.find_next("span"))
            if duration_tag else "Not satisfied"
        )

        if not pass_Filter(title, company, location, stipend, duration):
            rejected += 1
            continue

        cursor.execute("""
            INSERT INTO internships
            (title, company, location, stipend, duration, link)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (title, company, location, stipend, duration, link))

        saved += 1

        # ✅ Bug 1 Fix — result.append ab job loop ke ANDAR hai
        result.append({
            "title":    title,
            "company":  company,
            "location": location,
            "stipend":  stipend,
            "duration": duration,
            "link":     link,
        })

    time.sleep(2)  # ✅ sleep page loop mein hai, sahi jagah

cursor.execute("SELECT * FROM internships")
rows = cursor.fetchall()

for row in rows:
    print(row)

con.commit()
con.close()

print(f"After filtering: {len(result)} matched, {rejected} skipped\n")

if not result:
    print("No internship match with your filter")
else:
    for i, r in enumerate(result, start=1):
        print(f"[{i}] {r['title']}")
        print(f"    Company:  {r['company']}")
        print(f"    Location: {r['location']}")
        print(f"    Stipend:  {r['stipend']}")
        print(f"    Duration: {r['duration']}")
        print("-" * 45)

Page 1: 51 internships found
Page 2: 41 internships found
Page 3: 41 internships found
Page 4: 41 internships found
Page 5: 41 internships found
(267, 'Training Coordinator', 'Mind Appraisers', 'Hyderabad                                         (Hybrid)', '₹ 8,000 - 10,000 /month', '4 Months', 'https://internshala.com/internship/detail/training-coordinator-internship-in-hyderabad-at-mind-appraisers1778589931')
(268, 'Creative Writing', 'StoryMirror Infotech Private Limited', 'Mumbai', '₹ 5,000 - 10,000 /month', '3 Months', 'https://internshala.com/internship/detail/creative-writing-internship-in-mumbai-at-storymirror-infotech-private-limited1779859173')
(269, 'Business Development (Sales)', 'StoryMirror Infotech Private Limited', 'Mumbai', '₹ 10,000 - 15,000 /month', '3 Months', 'https://internshala.com/internship/detail/business-development-sales-internship-in-mumbai-at-storymirror-infotech-private-limited1779815911')
(270, 'Telecalling', 'MiTran Global', 'Chennai', '₹ 10,000 - 15,000

In [6]:
import csv
with open("Internship.csv","w",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=["title","company","location","duration","stipend","link"])
    writer.writeheader()
    writer.writerows(result)
print(f"\n Done ! saved {len(result)} internship in internship.csv")


 Done ! saved 215 internship in internship.csv
